# 11_second_round
Second-round referee responses.
- R1 Power / precision for the lagged HTN null: minimum detectable effect (MDE)
     and an equivalence (TOST-style) assessment on the odds-ratio scale, so an
     under-powered null is not read as evidence of absence.
- R2 Symmetry: run the lagged design for diabetes on the same footing as HTN.
- R3 Reconcile the negative control: test whether the ER-visit association
     itself survives the lagged design.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy import stats
rng=np.random.default_rng(42)
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
WAVE_SEQ=list(WAVES.keys())

def ipw(d,cols):
    Xp=d[cols].copy()
    for c in cols: Xp[c]=pd.to_numeric(Xp[c],errors="coerce"); Xp[c]=Xp[c].fillna(Xp[c].median())
    e=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=1000))]).fit(Xp,d["TREAT"]).predict_proba(Xp)[:,1]
    e=np.clip(e,0.02,0.98); pt=d["TREAT"].mean()
    return np.where(d["TREAT"]==1,pt/e,(1-pt)/(1-e))

def build_lagged(target, outcome_col=None, tau_q=0.75):
    """Lagged design: exposure t0->t1, outcome over t1->t2 among disease-free at t1.
       If outcome_col given (e.g. ER visit), use that binary at t2 instead of onset."""
    rows=[]
    for i in range(len(WAVE_SEQ)-2):
        w0,w1,w2=WAVE_SEQ[i],WAVE_SEQ[i+1],WAVE_SEQ[i+2]
        a=panel[panel.wave==w0].set_index(KEY); b=panel[panel.wave==w1].set_index(KEY); c=panel[panel.wave==w2].set_index(KEY)
        idx=a.index.intersection(b.index).intersection(c.index)
        df=pd.DataFrame(index=idx)
        df["bmi0"]=a.loc[idx,"BMI"]; df["bmi1"]=b.loc[idx,"BMI"]
        df["dx1"]=b.loc[idx,f"{target}_dx"]; df["dx2"]=c.loc[idx,f"{target}_dx"]
        df["age0"]=a.loc[idx,"age"]; df["male"]=(a.loc[idx,"SEX"]=="M").astype(int)
        df["inc0"]=a.loc[idx,"H_INC_TOT"]; df["year0"]=WAVES[w0]
        if outcome_col is not None:
            df["ncout"]=(pd.to_numeric(c.loc[idx,outcome_col],errors="coerce")>0).astype(float)
        rows.append(df.reset_index())
    L=pd.concat(rows,ignore_index=True).dropna(subset=["bmi0","bmi1","dx1","age0"])
    L=L[L["dx1"]==0]                          # disease-free at t1
    red=-(L["bmi1"]-L["bmi0"]); tau=float(red[red>0].quantile(tau_q))
    L["TREAT"]=(red>=tau).astype(int)
    if outcome_col is None:
        L=L.dropna(subset=["dx2"]); L["onset"]=(L["dx2"]==1).astype(int)
    else:
        L=L.dropna(subset=["ncout"]); L["onset"]=L["ncout"]
    L["inc0"]=L["inc0"].fillna(L["inc0"].median()); L["year0"]=L["year0"].astype("category")
    return L

def fit_or(L):
    sw=ipw(L,["bmi0","age0","male","inc0"])
    m=smf.glm("onset ~ TREAT+bmi0+age0+male+C(year0)",data=L,family=sm.families.Binomial(),freq_weights=sw).fit(cov_type="HC1")
    b=m.params["TREAT"]; se=m.bse["TREAT"]
    return np.exp(b),np.exp(m.conf_int().loc["TREAT"]).values,m.pvalues["TREAT"],b,se,int(L.TREAT.sum()),int(L.onset.sum())
print("helpers ready")

helpers ready


In [3]:
# R1: precision of the lagged HTN null.
orH,ciH,pH,bH,seH,ntH,noH = fit_or(build_lagged("HTN"))
# Minimum detectable OR at 80% power, alpha=0.05 two-sided, given the SE on log-OR:
z_a=stats.norm.ppf(0.975); z_b=stats.norm.ppf(0.80)
mde_logor=(z_a+z_b)*seH
mde_or_low=np.exp(-mde_logor); mde_or_high=np.exp(mde_logor)
# Equivalence (TOST) on log-OR against a margin. Use a clinically modest margin
# corresponding to OR bounds [0.80,1.25] (i.e. |log OR| <= ln 1.25).
margin=np.log(1.25)
# TOST: reject non-equivalence if CI for logOR within (-margin, +margin)
t_lower=(bH-(-margin))/seH; p_lower=1-stats.norm.cdf(t_lower)      # H0: logOR<=-margin
t_upper=((margin)-bH)/seH;  p_upper=1-stats.norm.cdf(t_upper)      # H0: logOR>=+margin
p_tost=max(p_lower,p_upper)
equiv = (np.exp(bH-z_a*seH)>=1/1.25) and (np.exp(bH+z_a*seH)<=1.25)
r1=pd.DataFrame([{
  "lagged_HTN_OR":round(orH,3),"ci_lo":round(ciH[0],3),"ci_hi":round(ciH[1],3),
  "SE_logOR":round(seH,3),"MDE_OR_at_80pct_power":f"{mde_or_low:.2f} or {mde_or_high:.2f}",
  "TOST_p_equiv_margin_1.25":round(p_tost,3),"equivalent_within_0.8_1.25":bool(equiv),
  "treated_n":ntH,"onset_n":noH}])
savetable(r1,"t11_r1_power_equivalence", index=False)
print(r1.T.to_string())

saved: t11_r1_power_equivalence.csv
                                       0
lagged_HTN_OR                       1.03
ci_lo                              0.779
ci_hi                              1.361
SE_logOR                           0.142
MDE_OR_at_80pct_power       0.67 or 1.49
TOST_p_equiv_margin_1.25           0.087
equivalent_within_0.8_1.25         False
treated_n                           1733
onset_n                              695


In [4]:
# R2: lagged design for diabetes, same footing as HTN
orD,ciD,pD,bD,seD,ntD,noD = fit_or(build_lagged("DM"))
r2=pd.DataFrame([
 {"target":"HTN","design":"lagged","OR":round(orH,3),"lo":round(ciH[0],3),"hi":round(ciH[1],3),"p":round(pH,3),"treated":ntH,"onset":noH},
 {"target":"DM","design":"lagged","OR":round(orD,3),"lo":round(ciD[0],3),"hi":round(ciD[1],3),"p":round(pD,3),"treated":ntD,"onset":noD},
])
savetable(r2,"t11_r2_lagged_both_targets", index=False)
print(r2.to_string(index=False))

saved: t11_r2_lagged_both_targets.csv
target design    OR    lo    hi     p  treated  onset
   HTN lagged 1.030 0.779 1.361 0.838     1733    695
    DM lagged 0.919 0.623 1.356 0.671     2243    392


In [5]:
# R3: does the negative-control (any ER visit) association survive the lagged design?
Lnc=build_lagged("HTN", outcome_col="ERGUN")   # ER visit count at t2 -> any-visit indicator
orN,ciN,pN,bN,seN,ntN,noN = fit_or(Lnc)
r3=pd.DataFrame([
 {"spec":"negative control, contemporaneous (from 07)","OR":1.364,"lo":1.177,"hi":1.582},
 {"spec":"negative control, lagged design","OR":round(orN,3),"lo":round(ciN[0],3),"hi":round(ciN[1],3)},
])
savetable(r3,"t11_r3_negcontrol_lagged", index=False)
print(r3.to_string(index=False))
print("\nInterpretation: if the lagged negative-control association attenuates toward 1,",
      "the healthcare-engagement confounding operates largely within the contemporaneous",
      "window, consistent with the lagged main-effect null not being manufactured by it.")

saved: t11_r3_negcontrol_lagged.csv
                                       spec    OR    lo    hi
negative control, contemporaneous (from 07) 1.364 1.177 1.582
            negative control, lagged design 1.376 1.165 1.624

Interpretation: if the lagged negative-control association attenuates toward 1, the healthcare-engagement confounding operates largely within the contemporaneous window, consistent with the lagged main-effect null not being manufactured by it.


In [6]:
fig,ax=plt.subplots(figsize=(6.4,3.0))
items=[("HTN onset (lagged)",orH,ciH),("DM onset (lagged)",orD,ciD),
       ("ER visit / neg-control (lagged)",orN,ciN)]
y=np.arange(len(items))[::-1]
for i,(lab,o,ci) in zip(y,items):
    ax.plot([ci[0],ci[1]],[i,i],color="#333333",lw=1.5); ax.plot(o,i,"o",color="#000000",ms=6)
ax.axvspan(0.8,1.25,color="#eeeeee",zorder=0)     # equivalence band
ax.axvline(1.0,color="#999999",lw=0.9,ls="--")
ax.set_yticks(y); ax.set_yticklabels([it[0] for it in items],fontsize=8)
ax.set_xlabel("Odds ratio (lagged design); shaded = equivalence band [0.80, 1.25]")
ax.set_xlim(0.5,1.7)
savefig(fig,"f11_lagged_forest"); plt.close(fig)
print("figure saved")

saved: f11_lagged_forest.png / f11_lagged_forest.pdf
figure saved
